# 11. MCP (Model Context Protocol)


## Learning Objectives

Learn how to connect external tools and context to an agent through MCP (Model Context Protocol).

This notebook covers:
- The MCP architecture: server, client, and host
- Connecting through the beta `langchain.mcp.MCPAdapter` API
- Injecting MCP tools with `MCPConfig` and `await adapter.list_tools()`
- The difference between stdio and streamable-http transports
- Extending multi-server MCP with authentication, elicitation, and middleware

## 11.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("Environment ready.")

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 11.2 MCP Concepts

**MCP (Model Context Protocol)** is an open protocol for providing external tools and context to an LLM in a standardized way. In LangChain 1.4, `langchain.mcp.MCPAdapter` is the beta surface that adapts FastMCP tools into LangChain tools.

| Component | Role | Example |
|----------|------|------|
| **MCP server** | Exposes tools, resources, and prompts | File system server, DB server, API wrapper |
| **MCP client** | Connects to the server and fetches tools | `MCPAdapter` / FastMCP `Client` |
| **Host** | Manages the client and connects it to the LLM | LangChain agent, IDE |

`MCPAdapter` currently focuses on tool conversion. MCP prompts and resources do not have one-to-one wrapper methods in `langchain.mcp`; read them directly through the FastMCP client when needed.

## 11.3 Installing `langchain[mcp]` and the Beta MCPAdapter

In LangChain 1.4, MCP tool integration uses the beta `langchain.mcp.MCPAdapter` surface. The runtime needs the `langchain[mcp]` extra plus the `mcp`/`fastmcp` packages required by the MCP server implementation.

The older separate adapter package is migration context only; new exercises use `MCPAdapter` with standard `MCPConfig`.

In [ ]:
# MCP beta adapter installation commands
print("MCP beta adapter installation:")
print("  uv add langchain[mcp] mcp")
print()
print("Key entry points:")
print("  - from langchain.mcp import MCPAdapter")
print("  - async with MCPAdapter(target) as adapter:")
print("        tools = await adapter.list_tools()")
print("  - target: URL, Path, FastMCP Client, ClientGroup, MCPConfig")
print("  - MCPConfig: {'mcpServers': {'name': {'command': ..., 'args': [...]}}}")
print("Call list_tools() inside the adapter context; returned tools retain client context.")

## 11.4 Stdio Transport

**Stdio (Standard I/O)** transport communicates with an MCP server through a local subprocess. It is a good fit for development and testing environments.


In [ ]:
from pathlib import Path
import json, tempfile, sys
server_path = Path(tempfile.gettempdir()) / "lc_mcp_math_server.py"
server_code = """from fastmcp import FastMCP
mcp = FastMCP("math")
@mcp.tool()
def add(a: int, b: int) -> int:
    return a + b
if __name__ == "__main__":
    mcp.run(transport="stdio")
"""
server_path.write_text(server_code)
stdio_config = {"mcpServers": {"math": {"command": sys.executable, "args": [str(server_path)]}}}
print("Stdio MCPConfig:")
print(json.dumps(stdio_config, indent=2, ensure_ascii=False))
print(f"\nServer file: {server_path}")

First verify the MCP server and adapter without a model call. This cell only performs tool discovery and direct tool invocation, so it is suitable for offline validation.

In [ ]:
from langchain.mcp import MCPAdapter

async with MCPAdapter(stdio_config) as adapter:
    tools = await adapter.list_tools()
    add_tool = next(t for t in tools if t.name.endswith("add"))
    result = await add_tool.ainvoke({"a": 2, "b": 3})
print([t.name for t in tools], result[0]["text"])

## 11.5 SSE / HTTP Transport

**HTTP (streamable-http)** transport uses web-based communication and is a good fit for remote MCP servers. It also supports authentication headers and custom settings.


In [ ]:
# Example HTTP / streamable-http configuration
http_config = {
    "mcpServers": {
        "weather": {"url": "https://weather-mcp.example.com/mcp"}
    }
}
import json
print("HTTP MCPConfig:")
print(json.dumps(http_config, indent=2))
print("\nUsage: async with MCPAdapter(http_config) -> await adapter.list_tools()")

### Authentication — let FastMCP Client own auth

For remote MCP servers, configure authentication on a FastMCP `Client`, then pass that client to `MCPAdapter`. This keeps token, OAuth, transport, and connection lifetime concerns together. If you need custom HTTP auth, use a FastMCP-compatible `httpx2.Auth` object rather than a regular `httpx.Auth` object.

| Mode | Example | Note |
|------|---------|------|
| Static token | `Client(url, auth=token)` | Read the token string from an environment variable in real code |
| OAuth | `Client(url, auth="oauth")` | FastMCP starts the OAuth flow; configure persistent token storage for production |
| Multiple servers | `ClientGroup({"name": Client(...)})` | Each server keeps its own auth settings |

Run tool discovery inside `async with MCPAdapter(client_or_group)`. The returned LangChain tools retain client context, so they can also be passed to an agent outside the block.

In [ ]:
# FastMCP Client / ClientGroup auth configuration example(no connection opened)
from fastmcp import Client, ClientGroup

weather_token = "${WEATHER_MCP_TOKEN}"
weather_client = Client("https://weather-mcp.example.com/mcp", auth=weather_token)
calendar_client = Client("https://calendar.example.com/mcp", auth="oauth")
auth_group = ClientGroup({"weather": weather_client, "calendar": calendar_client})
print(type(auth_group).__name__, list(auth_group.clients))

## 11.6 Loading MCP Tools and Integrating Them with an Agent

This is the common pattern for binding tools fetched from an MCP server into a LangChain agent.


In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter

async def run_math_agent():
    async with MCPAdapter(stdio_config) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent(model=model, tools=tools, system_prompt="You can use MCP math tools.")
        return await agent.ainvoke({"messages": [{"role": "user", "content": "Use the add tool to add 2 and 3."}]}, config=lf_config)

result = await run_math_agent()
print(result["messages"][-1].content)

## 11.7 Connecting to Multiple MCP Servers

Multiple servers go under the standard `mcpServers` key in `MCPConfig`. The new surface infers the transport from `command`/`args` or `url`, instead of requiring a separate `transport` field in each entry.

In [ ]:
# Example multi-MCP-server configuration
import json, sys
multi_server_config = {"mcpServers": {
    "math": {"command": sys.executable, "args": [str(server_path)]},
    "weather": {"url": "https://weather-mcp.example.com/mcp"},
    "database": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-postgres"], "env": {"DATABASE_URL": "postgresql://..."}},
}}
print("Multi-server MCPConfig:")
print(json.dumps(multi_server_config, indent=2, ensure_ascii=False))
print("Tool names are prefixed as server_tool for multi-server configs.")

## 11.7b Elicitation — server requests for user input

An MCP server can ask the user for more input while a tool is running. With `MCPAdapter` and a LangChain agent, a modern server's `InputRequiredResult` appears as a LangGraph interrupt. The older server-initiated `ctx.elicit(...)` push path cannot be answered by this interrupt loop with the current FastMCP/MCP stack, so the teaching example assumes an `InputRequiredResult`-compatible server.

In a `version="v2"` response, read `payload.requests[0].key`, then resume with an `accept`, `decline`, or `cancel` response under the same key.

| `action` | Meaning |
|----------|---------|
| `accept` | The user approved and provided values in `content` |
| `decline` | The user declined, so the server should choose an alternate path |
| `cancel` | The user cancelled the tool flow |

In [ ]:
# Elicitation — resume with payload.requests[0].key
elicit_example = '''
from langgraph.types import Command
first = await agent.ainvoke(input_state, config=config, version="v2")
payload = first.interrupts[0].value; key = payload.requests[0].key
resume = {"responses": {key: {"action": "accept", "content": {"value": "user input"}}}}
final = await agent.ainvoke(Command(resume=resume), config=config, version="v2")
'''
print(elicit_example)

## 11.8 Tool-call middleware

After MCP tools are converted to LangChain Tools, they use the same middleware surface as other tools. Keep authentication in the FastMCP `Client`, and put logging, policy checks, response filtering, and retries in LangChain middleware such as `@wrap_tool_call`.

| Use case | Description |
|----------|-------------|
| Call logging | Record tool calls that carry MCP metadata |
| Request policy | Check risky tool names or arguments |
| Response filtering | Remove sensitive content from a ToolMessage |
| Retry handling | Apply LangChain-side retry policy around tool exceptions |

In [ ]:
from collections.abc import Callable
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest

@wrap_tool_call
def log_mcp_calls(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage]) -> ToolMessage:
    mcp_meta = (request.tool.metadata or {}).get("mcp")
    if mcp_meta:
        print(f"MCP tool call: {request.tool_call['name']}")
    return handler(request)

async def run_with_middleware():
    async with MCPAdapter(stdio_config) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent(model=model, tools=tools, middleware=[log_mcp_calls])
        return await agent.ainvoke({"messages": [{"role": "user", "content": "Use the add tool to add 7 and 8."}]}, config=lf_config)

In [ ]:
# Tool-call policy belongs in LangChain middleware
middleware_note = '''
from langchain.agents.middleware import wrap_tool_call
@wrap_tool_call
def guard_mcp_tool(request, handler):
    if (request.tool.metadata or {}).get("mcp"):
        print(f"MCP policy check: {request.tool_call['name']}")
    return handler(request)
'''
print(middleware_note)

## 11.9b Tool lifetime — discovery and call scope

The earlier adapter package exposed session-binding helpers. With `langchain.mcp`, call `list_tools()` inside `async with MCPAdapter(...)`; the returned tools retain client context and can reconnect when invoked after the adapter block exits. For teaching examples, keeping discovery and one agent call in the same function still makes the lifetime easy to read. When a workflow depends on one server session's state or on concurrent calls through a shared client, manage that scope explicitly.

In [ ]:
# Discover tools in the adapter context, then pass them to the agent
stateful_example = '''
from langchain.mcp import MCPAdapter
async def run_with_adapter_lifetime():
    async with MCPAdapter(stdio_config) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent(model=model, tools=tools)
        return await agent.ainvoke({"messages": [{"role": "user", "content": "Add 2 and 3, then multiply the result by 5."}]})
'''
print(stateful_example)

## 11.10 Writing a Custom MCP Server

With the **FastMCP** library, you can build an MCP server quickly using decorators.


In [ ]:
from pathlib import Path
import tempfile, sys
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter

fastmcp_path = Path(tempfile.gettempdir()) / "lc_fastmcp_server.py"
fastmcp_path.write_text("""from fastmcp import FastMCP
mcp = FastMCP("my-tools")
@mcp.tool()
def add(a: int, b: int) -> int:
    return a + b
@mcp.tool()
def multiply(a: int, b: int) -> int:
    return a * b
if __name__ == "__main__":
    mcp.run(transport="stdio")
""")
async def run_custom_server():
    target = {"mcpServers": {"my_tools": {"command": sys.executable, "args": [str(fastmcp_path)]}}}
    async with MCPAdapter(target) as adapter:
        agent = create_agent(model=model, tools=await adapter.list_tools(), system_prompt="Use the multiply tool.")
        return await agent.ainvoke({"messages": [{"role": "user", "content": "Use the multiply tool to multiply 6 and 7."}]}, config=lf_config)
result = await run_custom_server(); print(result["messages"][-1].content)

## 11.11 Summary

This notebook covered:

| Topic | Key Idea |
|------|----------|
| **MCP concepts** | An open protocol that provides external tools and context to an LLM in a standardized way |
| **Stdio transport** | Local subprocess communication, good for development and testing |
| **Streamable HTTP** | Web-based communication for remote servers and authentication |
| **Agent integration** | Use `await adapter.list_tools()` inside `async with MCPAdapter(...)`, then pass tools to `create_agent` |
| **Multi-server support** | Configure several servers under `MCPConfig.mcpServers` or a FastMCP `ClientGroup` |
| **Authentication** | Configure bearer/OAuth on FastMCP `Client` objects before passing them to `MCPAdapter` |
| **Elicitation** | Server input requests are handled through interrupt/resume keyed by `payload.requests[0].key` |
| **Middleware** | Tool-call logging and policy belong in LangChain middleware such as `@wrap_tool_call` |
| **Lifetime** | Call `list_tools()` inside the adapter context; returned tools retain client context |

### Next Steps
- Continue to frontend streaming.

---
**References:**
- [MCP (Model Context Protocol)](../docs/langchain/16-mcp.md)
